# Clase 2 — Matplotlib: gráficos de líneas y áreas

## ¿Por qué te importa esto?

Una de las preguntas más comunes en cualquier análisis de datos es: "¿cómo ha cambiado esto a lo largo del tiempo?". ¿Suben las ventas? ¿Bajan los casos? ¿Se disparó algo en un año concreto? El **gráfico de líneas** es la respuesta visual más directa a ese tipo de pregunta, y es probablemente el primer gráfico que vas a dibujar en cualquier proyecto real.

En esta clase vamos a usar `pandas` y `matplotlib` para dibujar tendencias con el dataset de inmigración a Canadá. Vamos a ver un solo país, varios países a la vez, y una variante del gráfico de líneas — el **gráfico de área** — que además de mostrar la tendencia, rellena el espacio bajo la línea para dar sensación de volumen acumulado.

## Dos formas de dibujar: la nota rápida y el diseño por capas

Matplotlib te ofrece dos maneras de trabajar, y conviene que entiendas la diferencia desde el principio:

- **Scripting layer** (`plt.plot()`, o directamente `df.plot()` desde pandas): es como escribir una nota rápida a mano. Le dices "dibuja esto" y matplotlib decide por ti el resto — qué figura usar, dónde ponerla. Es rápido y perfecto para explorar datos.
- **Artist layer** (`fig, ax = plt.subplots()`): es como diseñar con capas superpuestas en un programa de diseño, donde tú controlas explícitamente cada elemento — el lienzo (`fig`), los ejes (`ax`), qué se dibuja en cada uno. Es más código, pero te da control total, especialmente cuando quieres varios gráficos en la misma figura.

Ambos enfoques producen resultados válidos. Vamos a ver ejemplos de los dos a lo largo de la clase.

## Preparar el entorno y cargar los datos

Cargamos el dataset de inmigración a Canadá (el mismo que usamos en la clase de pandas), lo indexamos por país y preparamos la lista de años.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

%matplotlib inline

print('Matplotlib version: ', mpl.__version__)

In [ ]:
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Canada.csv'

df_can = pd.read_csv(URL)

print('Data read into a pandas dataframe!')
df_can.head()

Como vimos en la clase anterior, conviene tener el DataFrame indexado por país y la lista de años como texto lista para usar en cualquier gráfico.

In [ ]:
df_can.set_index('Country', inplace=True)
df_can.index.name = None

years = list(map(str, range(1980, 2014)))

df_can.head(3)

## Line plot de un solo país (una Serie)

Empecemos con un caso real: en 2010, Haití sufrió un terremoto devastador de magnitud 7.0. Como parte de su respuesta humanitaria, Canadá aceptó muchos más refugiados haitianos ese año. Vamos a visualizarlo.

Primero extraemos los datos de Haití como una **Serie** (una sola fila, con los años como índice):

In [ ]:
haiti = df_can.loc['Haiti', years]  # serie: índice = años, valores = número de inmigrantes
haiti.head()

Con el enfoque de scripting layer, graficar una Serie es tan simple como llamar a `.plot()`. `pandas` puebla automáticamente el eje X con los valores del índice (los años) y el eje Y con los valores de la columna.

In [ ]:
haiti.plot(kind='line')

plt.title('Inmigración desde Haití')
plt.ylabel('Número de inmigrantes')
plt.xlabel('Años')

plt.show()  # necesario para mostrar los cambios hechos a la figura

Los años están guardados como texto (`str`), lo que puede afectar cómo se ven los ticks del eje X. Para un resultado más limpio, conviene convertir el índice a entero antes de graficar. Aprovechamos para anotar el terremoto de 2010 directamente sobre el gráfico con `plt.text(x, y, etiqueta)`.

In [ ]:
haiti.index = haiti.index.map(int)
haiti.plot(kind='line')

plt.title('Inmigración desde Haití')
plt.ylabel('Número de Inmigrantes')
plt.xlabel('Años')

# anotamos el terremoto de 2010 directamente sobre el gráfico
# sintaxis: plt.text(x, y, etiqueta)
plt.text(2000, 6000, '2010 Earthquake')

plt.show()

**Nota sobre `plt.text(x, y, etiqueta)`**: como el eje X (años) ahora es de tipo entero, `x` se especifica como un año real (por ejemplo, 2000). Si el índice siguiera siendo texto, tendríamos que usar la posición del año en la lista en lugar del año mismo.

## Line plot con varios países (un DataFrame)

Ahora comparemos la inmigración de India y China entre 1980 y 2013.

In [ ]:
df_CI = df_can.loc[['India', 'China'], years]
df_CI

Si intentamos graficar `df_CI` directamente con `.plot(kind='line')`, el resultado no se ve bien. La razón: pandas grafica los **valores del índice en el eje X** y **cada columna como una línea distinta**. Como `df_CI` tiene los países en el índice y los años en las columnas, pandas intenta dibujar una línea por cada año — justo al revés de lo que queremos.

La solución es **transponer** el DataFrame con `.transpose()`, que intercambia filas por columnas.

In [ ]:
df_CI = df_CI.transpose()
df_CI.head()

Ahora los años quedan en el índice (eje X) y cada país es una columna (una línea). Convertimos el índice a entero y graficamos.

In [ ]:
df_CI.index = df_CI.index.map(int)
df_CI.plot(kind='line')

plt.title('Inmigrantes desde China e India')
plt.ylabel('Número de Inmigrantes')
plt.xlabel('Años')

plt.show()

**¿Por qué no tuvimos que transponer `haiti` antes?** Porque `haiti` es una **Serie**, no un DataFrame: ya tiene los años como índice de forma natural, con un único valor por año. Un DataFrame en cambio tiene filas y columnas, y `.plot()` siempre asume que el índice va al eje X y cada columna se convierte en una línea — de ahí que solo necesites transponer cuando tus "series a comparar" están organizadas como filas en lugar de columnas.

Como buena práctica: evita meter más de 5 a 10 líneas en un mismo gráfico de líneas. Más allá de eso, se vuelve difícil de leer.

## Top 5 países con más inmigración

Combinemos `sort_values()`, `.head()` y `.transpose()` para comparar la tendencia de los 5 países que más aportaron a la inmigración canadiense.

In [ ]:
df_top5 = df_can.sort_values(by='Total', ascending=False, axis=0).head(5)
df_top5 = df_top5[years].transpose()
df_top5.index = df_top5.index.map(int)

df_top5.plot(kind='line', figsize=(14, 8))

plt.title('Tendencia de Inmigración de los 5 Países Principales')
plt.ylabel('Número de Inmigrantes')
plt.xlabel('Años')

plt.show()

## El equivalente en la artist layer

El mismo gráfico de línea, pero controlando explícitamente figura y ejes con `fig, ax = plt.subplots()`.

In [ ]:
# creamos la figura y los ejes
fig, ax = plt.subplots()

total_immigrants = df_can[years].sum()          # total de inmigrantes por año
total_immigrants.index = total_immigrants.index.map(int)

ax.plot(total_immigrants)

ax.set_title('Inmigrantes entre 1980 y 2013')
ax.set_xlabel('Años')
ax.set_ylabel('Total de Inmigrantes')

plt.show()

Con `ax.plot()` podemos personalizar la línea con más detalle: marcador, tamaño, color y estilo de línea.

In [ ]:
fig, ax = plt.subplots()

ax.plot(total_immigrants,
        marker='s',           # marcadores en forma de cuadrado
        markersize=5,         # tamaño del marcador
        color='green',        # color de la línea
        linestyle='dotted')   # línea punteada

ax.set_title('Inmigrantes entre 1980 y 2013')
ax.set_xlabel('Años')
ax.set_ylabel('Total de Inmigrantes')
ax.legend(['Inmigrantes'])

plt.show()

## Area plot

El gráfico de área es prácticamente un gráfico de líneas con el espacio bajo la curva relleno de color, lo que ayuda a comunicar volumen acumulado en vez de solo la tendencia. En pandas se usa `kind='area'`.

Por defecto, `pandas` apila las áreas unas sobre otras (`stacked=True`), de modo que el total visual en cada punto es la suma de todas las series.

In [ ]:
df_top5.plot(kind='area', alpha=0.5, figsize=(14, 8))  # alpha controla la transparencia, stacked=True por defecto

plt.title('Tendencia de Inmigración de los 5 Países Principales (apilado)')
plt.ylabel('Número de Inmigrantes')
plt.xlabel('Años')

plt.show()

Si preferimos ver cada área superpuesta desde cero (para comparar tendencias individuales sin que se sumen), usamos `stacked=False`. En ese caso, `alpha` es especialmente útil porque las áreas se superponen y necesitamos ver a través de ellas.

In [ ]:
df_top5.plot(kind='area', stacked=False, alpha=0.5, figsize=(14, 8))

plt.title('Tendencia de Inmigración de los 5 Países Principales (sin apilar)')
plt.ylabel('Número de Inmigrantes')
plt.xlabel('Años')

plt.show()

En la artist layer, el equivalente directo de un área es `ax.fill_between(x, y1, y2, color=..., alpha=...)`, que rellena el espacio entre dos curvas (o entre una curva y el eje X si `y2=0`).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.fill_between(total_immigrants.index, total_immigrants, color='skyblue', alpha=0.4)
ax.plot(total_immigrants, color='steelblue')

ax.set_title('Total de Inmigrantes entre 1980 y 2013 (area con fill_between)')
ax.set_xlabel('Años')
ax.set_ylabel('Total de Inmigrantes')
ax.grid(True)

plt.show()

## Personalización básica que vas a usar siempre

Ya viste varias de estas piezas en los ejemplos anteriores. Aquí las tienes juntas como referencia rápida:

```python
plt.title('Título del gráfico')      # o ax.set_title(...)
plt.xlabel('Etiqueta eje X')         # o ax.set_xlabel(...)
plt.ylabel('Etiqueta eje Y')         # o ax.set_ylabel(...)
plt.legend(['Serie 1', 'Serie 2'])   # o ax.legend([...])
plt.grid(True)                       # o ax.grid(True) — añade una cuadrícula de fondo
plt.text(x, y, 'Anotación')          # o ax.annotate('Anotación', xy=(x, y)) — texto en un punto específico
```

Con la artist layer, `ax.set_xlim()` te permite ajustar los límites del eje X, y `ax.set_xticks(...)` controla exactamente qué marcas aparecen en él — útil cuando el eje por defecto se ve saturado o poco claro.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(total_immigrants, marker='o', markersize=4, color='darkorange')

ax.set_title('Inmigrantes entre 1980 y 2013')
ax.set_xlabel('Años')
ax.set_ylabel('Total de Inmigrantes')
ax.legend(['Inmigrantes'])
ax.grid(True)
ax.set_xlim(1978, 2015)
ax.set_xticks(list(range(1980, 2015, 5)))

plt.show()

Este mismo dataset y esta misma librería sirven también para gráficos de dispersión, barras, histogramas y pastel — pero esos tipos de gráfico se cubren en detalle en otra clase dedicada a comparar categorías y distribuciones, no aquí.

## Puntos clave

- El scripting layer (`plt.plot()`, `df.plot()`) es rápido para explorar; la artist layer (`fig, ax = plt.subplots()`) te da control total, especialmente con múltiples gráficos.
- Una Serie ya tiene el eje X natural en su índice; un DataFrame necesita que las categorías a comparar estén en las columnas, por eso a veces hay que `.transpose()` antes de graficar.
- Convertir el índice de años a entero (`.map(int)`) evita problemas de visualización en el eje X.
- El area plot es un line plot con relleno bajo la curva; `stacked` decide si las áreas se suman o se superponen, y `alpha` controla la transparencia.
- `plt.title`, `plt.xlabel`, `plt.ylabel`, `plt.legend`, `plt.grid` y `plt.text`/`ax.annotate` son las piezas de personalización que vas a usar en casi cualquier gráfico.